## Milestone 2: NLP Data Preprocessing
We created this notebook to change our sensor data from the first milestone into a text format. This will allow us to use two different NLP models for our project:
•	Transformer Model: This uses positional encoding to understand the sequence.
•	BiLSTM Model: This uses learned embeddings for the tokens.
Input Files: We are reading the cleaned data that we processed earlier:
•	train_clean.pkl 
•	test_clean.pkl 
Output Files: After we finish the preprocessing, we will save these files to the data folder:
•	train_text_seq.pkl and test_text_seq.pkl: These have the ID sequences and labels.
•	nlp_vocab.pkl: This is the mapping we made for tokens to IDs.
•	nlp_feature_thresholds.pkl: We saved the bin boundaries for the sensors.
•	nlp_max_seq_len.pkl: This helps us keep the padding consistent for both models.


## 1. Imports
We start by importing the libraries we need, such as pandas and numpy, for handling our data and files. We also set the path for where we will save our results.

In [ ]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

DATA_PROCESSED = Path("../data/processed")

print("Imports completed successfully.")

Imports completed successfully.


## 2. Loading Milestone 1 Data
We are loading the cleaned files that we saved from our first preprocessing notebook. This includes both the training and testing sets so we can keep the work consistent.


In [ ]:
with open(DATA_PROCESSED / "train_clean.pkl", "rb") as f:
    train_df = pickle.load(f)

with open(DATA_PROCESSED / "test_clean.pkl", "rb") as f:
    test_df = pickle.load(f)

print(f"Train shape    : {train_df.shape}")
print(f"Test shape     : {test_df.shape}")
print(f"Train sequences: {train_df['sequence_id'].nunique()}")
print(f"Test sequences : {test_df['sequence_id'].nunique()}")

Train shape    : (460717, 341)
Test shape     : (113993, 341)
Train sequences: 6515
Test sequences : 1629


## 3. Constants and BFRB Gestures
We need to define which columns are for sequences and which ones are the gesture labels. We also made a list of the specific BFRB gestures we are looking for, such as skin picking or hair pulling.


In [ ]:
SEQUENCE_COL = "sequence_id"
GESTURE_COL  = "gesture"

BFRB_GESTURES = [
    "Cheek - pinch skin",
    "Forehead - pull hairline",
    "Neck - scratch",
    "Neck - pinch skin",
    "Eyelash - pull hair",
    "Eyebrow - pull hair",
    "Forehead - scratch",
    "Above ear - pull hair",
    "Pinch knee/leg skin",
    "Scratch knee/leg skin"
]

print(f"BFRB gesture classes: {len(BFRB_GESTURES)}")

BFRB gesture classes: 10


## 4. Creating Binary Labels
We are making new labels where 1 means the gesture is a BFRB and 0 means it is not. We checked the distribution to see how many of each label we have in our data.


In [ ]:
train_df["label"] = train_df[GESTURE_COL].apply(lambda x: 1 if x in BFRB_GESTURES else 0)
test_df["label"]  = test_df[GESTURE_COL].apply(lambda x: 1 if x in BFRB_GESTURES else 0)

print("Train label distribution:")
print(train_df["label"].value_counts())
print("\nTest label distribution:")
print(test_df["label"].value_counts())

Train label distribution:
label
1    294387
0    166330
Name: count, dtype: int64

Test label distribution:
label
1    71739
0    42254
Name: count, dtype: int64


## 5. Selecting Features for NLP
We decided to use only the 7 IMU channels and the 5 thermopile channels for this part. We are not using the TOF data here because it would make the vocabulary too big to handle easily.


In [ ]:
imu_cols     = ["acc_x", "acc_y", "acc_z", "rot_w", "rot_x", "rot_y", "rot_z"]
thm_cols     = [f"thm_{i}" for i in range(1, 6)]
feature_cols = imu_cols + thm_cols

print(f"NLP feature columns: {len(feature_cols)}")
print(feature_cols)

NLP feature columns: 12
['acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z', 'thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']


## 6. Bin Thresholds
We are splitting each sensor channel into three categories, which are low, medium, and high. We only used the training set to find these boundaries so we do not have any data leakage.


In [ ]:
feature_thresholds = {}

for col in feature_cols:
    q33 = train_df[col].quantile(0.33)
    q66 = train_df[col].quantile(0.66)
    feature_thresholds[col] = (q33, q66)

print("Bin thresholds computed (train set only).")
print("\nExample thresholds:")
for col in feature_cols[:4]:
    q33, q66 = feature_thresholds[col]
    print(f"  {col}: low <= {q33:.4f}, medium <= {q66:.4f}, high > {q66:.4f}")

Bin thresholds computed (train set only).

Example thresholds:
  acc_x: low <= -0.8008, medium <= 5.2812, high > 5.2812
  acc_y: low <= -1.2305, medium <= 4.4375, high > 4.4375
  acc_z: low <= -4.2383, medium <= 2.5469, high > 2.5469
  rot_w: low <= 0.2329, medium <= 0.4398, high > 0.4398


## 7. Tokenization Functions
We wrote some functions to turn the continuous sensor numbers into text tokens. For every timestamp, we get a list of tokens that describe what the sensors were doing at that moment.


In [ ]:
def value_to_bin(value, q33, q66):
    """Map a continuous sensor value to a discrete bin label."""
    if value <= q33:
        return "low"
    elif value <= q66:
        return "medium"
    else:
        return "high"


def sequence_to_tokens(group):
    """
    Convert a gesture sequence (DataFrame group) into a list of token strings.

    Each timestep produces one token per sensor channel.
    Returns a flat list of token strings across all timesteps.

    Example: ['acc_x_low', 'acc_y_high', 'thm_1_medium', ..., 'acc_x_medium', ...]

    This flat token list is what both the Transformer and BiLSTM models
    will receive as their input sequence.
    """
    all_tokens = []
    for _, row in group.iterrows():
        for col in feature_cols:
            q33, q66 = feature_thresholds[col]
            bin_label = value_to_bin(row[col], q33, q66)
            all_tokens.append(f"{col}_{bin_label}")
    return all_tokens


print("Tokenization functions defined.")

Tokenization functions defined.


## 8. Building the Vocabulary
We are making a list of all unique tokens found in our training data. We also added special tokens for padding and for any words that we might not recognize later.


In [ ]:
print("Building vocabulary from training set...")

all_train_tokens = []
for seq_id, group in train_df.groupby(SEQUENCE_COL):
    all_train_tokens.extend(sequence_to_tokens(group))

token_counts = Counter(all_train_tokens)

vocab = {"<PAD>": 0, "<UNK>": 1}
for token, _ in token_counts.most_common():
    vocab[token] = len(vocab)

print(f"Vocabulary size        : {len(vocab)} (including <PAD> and <UNK>)")
print(f"Unique tokens          : {len(token_counts)}")
print(f"Total token occurrences: {len(all_train_tokens):,}")

Building vocabulary from training set...
Vocabulary size        : 38 (including <PAD> and <UNK>)
Unique tokens          : 36
Total token occurrences: 5,528,604


## 9. Sequence to ID Conversion
We changed every gesture sequence into a list of integer IDs that the models can read. We also found the longest sequence in our data so we can pad all the other sequences to the same size.


In [ ]:
def build_id_dataset(df):
    """
    Convert all gesture sequences to token ID lists.
    Returns a list of dicts with keys: sequence_id, token_ids, label.
    """
    records = []
    for seq_id, group in df.groupby(SEQUENCE_COL):
        tokens   = sequence_to_tokens(group)
        token_ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
        label    = group["label"].iloc[0]
        records.append({
            "sequence_id": seq_id,
            "token_ids": token_ids,
            "label": label
        })
    return records


print("Converting train sequences...")
train_text_seq = build_id_dataset(train_df)
print("Converting test sequences...")
test_text_seq  = build_id_dataset(test_df)

# Compute max sequence length from training set (used for padding in model notebooks)
max_seq_len = max(len(r["token_ids"]) for r in train_text_seq)

print(f"\nTrain sequences : {len(train_text_seq)}")
print(f"Test sequences  : {len(test_text_seq)}")
print(f"Max seq length  : {max_seq_len} tokens")
print(f"Min seq length  : {min(len(r['token_ids']) for r in train_text_seq)} tokens")
print(f"\nExample first 10 token IDs: {train_text_seq[0]['token_ids'][:10]}")
print(f"Example label             : {train_text_seq[0]['label']}")

Converting train sequences...
Converting test sequences...

Train sequences : 6515
Test sequences  : 1629
Max seq length  : 8400 tokens
Min seq length  : 432 tokens

Example first 10 token IDs: [13, 11, 12, 20, 25, 21, 22, 4, 3, 5]
Example label             : 1


## 10. Saving Final Files
Finally, we saved all our token sequences and the vocabulary mapping as pickle files. We also saved the thresholds and the max length so we can use them again in the model notebooks.


In [ ]:
# Token ID sequences — used by both 07_model_nlp_transformer and 08_model_nlp_bilstm
with open(DATA_PROCESSED / "train_text_seq.pkl", "wb") as f:
    pickle.dump(train_text_seq, f)
with open(DATA_PROCESSED / "test_text_seq.pkl", "wb") as f:
    pickle.dump(test_text_seq, f)
print("Saved: train_text_seq.pkl")
print("Saved: test_text_seq.pkl")

# Vocabulary — used by both model notebooks for embedding layer
with open(DATA_PROCESSED / "nlp_vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)
print("Saved: nlp_vocab.pkl")

# Thresholds — saved for reproducibility
with open(DATA_PROCESSED / "nlp_feature_thresholds.pkl", "wb") as f:
    pickle.dump(feature_thresholds, f)
print("Saved: nlp_feature_thresholds.pkl")

# Max sequence length — used for consistent padding across both models
with open(DATA_PROCESSED / "nlp_max_seq_len.pkl", "wb") as f:
    pickle.dump(max_seq_len, f)
print("Saved: nlp_max_seq_len.pkl")

print("\n=== NLP Preprocessing Complete ===")
print(f"Train sequences  : {len(train_text_seq)}")
print(f"Test sequences   : {len(test_text_seq)}")
print(f"Vocabulary size  : {len(vocab)}")
print(f"Max seq length   : {max_seq_len}")
print(f"Feature channels : {len(feature_cols)} (IMU + thermopile only)")
print(f"Bins per channel : 3 (low / medium / high)")

Saved: train_text_seq.pkl
Saved: test_text_seq.pkl
Saved: nlp_vocab.pkl
Saved: nlp_feature_thresholds.pkl
Saved: nlp_max_seq_len.pkl

=== NLP Preprocessing Complete ===
Train sequences  : 6515
Test sequences   : 1629
Vocabulary size  : 38
Max seq length   : 8400
Feature channels : 12 (IMU + thermopile only)
Bins per channel : 3 (low / medium / high)
